# 🛒 Large-Scale E-Commerce Sentiment Analysis
## Production-Quality TF-IDF + LinearSVC Pipeline

**Dataset**: ~914k cleaned e-commerce reviews (positive / negative / neutral)
**Objective**: Maximize Macro F1 and Neutral-class F1, ensure memory-safe execution on Colab.
**Approach**: Advanced FeatureUnion (Word + Char TF-IDF) with Balanced LinearSVC.
**Environment**: Google Colab (Tesla T4 GPU, ~12 GB RAM)


## 1. Environment Setup & Memory Diagnostics


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import gc
import os
import time
import textwrap
import psutil

import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.15)

def get_memory_usage():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 ** 2

print("✅ Libraries imported successfully.")
print(f"🧠 Initial Memory usage: {get_memory_usage():.1f} MB")


## 2. Dataset Loading

Load the dataset optimally by defining specific data types to reduce peak RAM usage.


In [ ]:
CSV_PATH = "sentiment_final_cleaned.csv"

# Using 'category' dtype for sentiment and 'str' for text ensures minimal memory usage.
df = pd.read_csv(
    CSV_PATH,
    dtype={"text": "str", "sentiment": "category"},
    engine="c",
    low_memory=True,
)

print(f"✅ Loaded {len(df):,} rows × {df.shape[1]} columns")
print(f"🧠 DataFrame memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"🧠 System Memory usage: {get_memory_usage():.1f} MB")


## 3. Dataset Profiling & Class Imbalance Analysis


In [ ]:
print("── Shape ──")
print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}\n")

print("── Class distribution ──")
print(df["sentiment"].value_counts())
print()

# Text length statistics
lengths = df["text"].str.len()
print(f"Mean text length : {lengths.mean():.1f} chars")
print(f"Median text length: {lengths.median():.1f} chars")
print(f"Max text length  : {lengths.max():,} chars\n")

# Nulls and duplicates (expecting very few since dataset is cleaned)
print(f"Null counts:\n{df.isnull().sum()}\n")
print(f"Duplicates: {df.duplicated().sum():,}")
del lengths; gc.collect()


In [ ]:
# Class distribution visualization
fig, ax = plt.subplots(figsize=(8, 5))
counts = df["sentiment"].value_counts()
colors = ["#2ecc71", "#e74c3c", "#f39c12"]

bars = ax.bar(counts.index.astype(str), counts.values, color=colors, edgecolor="black", linewidth=0.5)
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 5000,
            f"{b.get_height():,}", ha="center", va="bottom", fontweight="bold")

ax.set_title("Sentiment Class Distribution (Imbalanced)", fontsize=14, fontweight="bold")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()
del counts; gc.collect()


## 4. Label Encoding
Encode categorical sentiments into integers safely.


In [ ]:
le = LabelEncoder()
y = le.fit_transform(df["sentiment"])
print(f"Classes mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"Encoded label dtype: {y.dtype}  |  Unique labels: {np.unique(y)}")


## 5. Efficient Train-Validation-Test Split

We perform a 80/10/10 split stratified by class to guarantee proper evaluation across all sets, especially for the minority neutral class.


In [ ]:
X_text = df["text"].fillna("").values  # Numpy array of strings (no memory copy)

# 1st split: Extract Test set (10%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_text, y, test_size=0.10, random_state=42, stratify=y
)

# 2nd split: Split remaining 90% into Train (80% overall) and Val (10% overall)
# 10/90 = 0.1111 for validation size
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1111, random_state=42, stratify=y_temp
)

print(f"Train set      : {len(X_train):,} samples")
print(f"Validation set : {len(X_val):,} samples")
print(f"Test set       : {len(X_test):,} samples\n")

# Verify stratifications
print(f"Train class dist: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Val   class dist: {dict(zip(*np.unique(y_val, return_counts=True)))}")

# Free original dataframe to save RAM
del df, X_text, y, X_temp, y_temp
gc.collect()
print(f"🧠 System Memory usage after split: {get_memory_usage():.1f} MB")


## 6. Advanced TF-IDF Feature Engineering

Optimized for e-commerce reviews:
- **Word TF-IDF**: Captures standard vocabulary (unigrams & bigrams).
- **Character TF-IDF**: Extremely robust to typos, misspellings, and morphological variations common in e-commerce.
- Both use `sublinear_tf=True` (logarithmic term frequency) to reduce the dominance of repeated words.
- Reduced `max_df` and optimized `min_df` to shrink vocabulary size and improve generalizability.


In [ ]:
word_tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=200000,
    sublinear_tf=True,
    min_df=10,        # Word must appear in at least 10 documents
    max_df=0.85,      # Ignore words appearing in >85% of documents
    dtype=np.float32, # Half-precision to save memory
    strip_accents="unicode",
)

char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5), # Character 3 to 5-grams inside word boundaries
    max_features=150000,
    sublinear_tf=True,
    min_df=15,
    max_df=0.85,
    dtype=np.float32,
    strip_accents="unicode",
)

# Combine both feature sets
vectorizer = FeatureUnion(
    [("word", word_tfidf), ("char", char_tfidf)],
    n_jobs=1, # Keep single-threaded to avoid memory spikes in Colab
)

print("✅ FeatureUnion defined (Word + Char_wb TF-IDF)")


## 7. Sparse Matrix Optimization & Vectorization
Fit and transform the training data, then transform validation and test sets.


In [ ]:
print("Fitting TF-IDF on training data ...")
t0 = time.time()
X_train_tfidf = vectorizer.fit_transform(X_train)
fit_time = time.time() - t0
print(f"✅ Train TF-IDF done in {fit_time:.1f}s")
print(f"   Shape: {X_train_tfidf.shape} | dtype: {X_train_tfidf.dtype}")
print(f"   Sparse matrix size: {X_train_tfidf.data.nbytes / 1e6:.1f} MB")
print(f"🧠 Memory: {get_memory_usage():.1f} MB\n")

print("Transforming validation and test data ...")
t0 = time.time()
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)
print(f"✅ Transformation done in {time.time()-t0:.1f}s")
print(f"   Val Shape: {X_val_tfidf.shape}")
print(f"   Test Shape: {X_test_tfidf.shape}")

# Free raw text arrays as they are no longer needed for training
del X_train, X_val
gc.collect()
print(f"🧠 Memory after cleanup: {get_memory_usage():.1f} MB")


## 8. LinearSVC Training with Balanced Weights

Linear Support Vector Classification is robust for large-scale, highly sparse datasets.
- `class_weight='balanced'` is critical here to boost the penalty for misclassifying the minority 'neutral' class.
- `C=0.5` acts as regularization to prevent overfitting on the very wide feature space (350k+ features).
- `dual="auto"` ensures optimal solver selection for latest scikit-learn versions.


In [ ]:
# Note: If running on scikit-learn < 1.2, you can safely use dual=False
svc = LinearSVC(
    C=0.5,
    class_weight="balanced", # Heavily penalizes errors on the neutral class
    max_iter=5000,           # Generous iterations for convergence
    random_state=42,
    loss="squared_hinge",
    dual="auto",             # Automatically chooses dual vs primal based on n_samples and n_features
)

print("Training LinearSVC ...")
t0 = time.time()
svc.fit(X_train_tfidf, y_train)
train_time = time.time() - t0
print(f"✅ Training complete in {train_time:.1f}s")
print(f"🧠 Memory after training: {get_memory_usage():.1f} MB")


## 9. Model Evaluation

Focusing on Weighted F1, Macro F1, and Neutral Class performance.


In [ ]:
y_pred_val = svc.predict(X_val_tfidf)
y_pred_test = svc.predict(X_test_tfidf)

def evaluate_predictions(y_true, y_pred, dataset_name="Test"):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted")
    rec  = recall_score(y_true, y_pred, average="weighted")
    f1_wt= f1_score(y_true, y_pred, average="weighted")
    f1_mac= f1_score(y_true, y_pred, average="macro")

    print("=" * 50)
    print(f"         {dataset_name.upper()} SET EVALUATION")
    print("=" * 50)
    print(f"  Accuracy          : {acc:.4f}")
    print(f"  Precision (wt)    : {prec:.4f}")
    print(f"  Recall    (wt)    : {rec:.4f}")
    print(f"  F1-score  (wt)    : {f1_wt:.4f}")
    print(f"  F1-score  (macro) : {f1_mac:.4f}  <-- Crucial for imbalanced data")
    print("=" * 50)

evaluate_predictions(y_val, y_pred_val, "Validation")
evaluate_predictions(y_test, y_pred_test, "Test")


In [ ]:
print("\n── Classification Report (Test Set) ──\n")
target_names = [str(c) for c in le.classes_]
print(classification_report(y_test, y_pred_test, target_names=target_names, digits=4))


## 10. Confusion Matrix Visualization


In [ ]:
cm = confusion_matrix(y_test, y_pred_test)

fig, ax = plt.subplots(figsize=(8, 6))
# Normalize the confusion matrix over true labels for better insight into class-wise accuracy
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names, ax=ax,
            linewidths=0.8, linecolor="white", cbar=False)

# Add normalized percentages below the raw counts
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j+0.5, i+0.7, f"({cm_normalized[i,j]:.1%})",
                ha="center", va="center", color="black" if cm_normalized[i,j] < 0.5 else "white",
                fontsize=9)

ax.set_xlabel("Predicted Label", fontsize=12, fontweight="bold")
ax.set_ylabel("True Label", fontsize=12, fontweight="bold")
ax.set_title("Confusion Matrix (with Recall Percentages)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 11. Prediction Confidence & Decision Margin Analysis

LinearSVC does not output probabilities directly, but its `decision_function` gives a signed distance to the hyperplane. This acts as a proxy for prediction confidence.


In [ ]:
decision_scores = svc.decision_function(X_test_tfidf)

# Plot histogram of max decision scores (confidence proxy)
max_scores = np.max(decision_scores, axis=1)

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(max_scores, bins=50, kde=True, color="#3498db", ax=ax)
ax.set_title("Distribution of Prediction Confidence (Decision Margin)", fontweight="bold")
ax.set_xlabel("Max Decision Function Score")
ax.set_ylabel("Frequency")
plt.axvline(np.median(max_scores), color='red', linestyle='dashed', label=f'Median: {np.median(max_scores):.2f}')
plt.legend()
plt.show()


## 12. Misclassification & Error Analysis

Where is the model failing? We look specifically at True=Neutral errors since that is the hardest class.


In [ ]:
# Create a DataFrame for test errors
df_err = pd.DataFrame({
    "text": X_test,
    "true_label": le.inverse_transform(y_test),
    "pred_label": le.inverse_transform(y_pred_test),
    "confidence": max_scores
})

errors = df_err[df_err["true_label"] != df_err["pred_label"]]
print(f"Total test samples : {len(df_err):,}")
print(f"Misclassified      : {len(errors):,} ({len(errors)/len(df_err)*100:.2f}%)\n")

print("── Top 5 Confusion Pairs ──")
err_pairs = errors.groupby(["true_label", "pred_label"]).size().sort_values(ascending=False)
print(err_pairs.head(5))


In [ ]:
print("\n── Sample Misclassified Reviews (True: neutral) ──\n")
neutral_errors = errors[errors["true_label"] == "neutral"].sort_values(by="confidence", ascending=False)

for pred_class in [c for c in le.classes_ if c != "neutral"]:
    subset = neutral_errors[neutral_errors["pred_label"] == pred_class]
    if not subset.empty:
        print(f"► TRUE: neutral  →  PREDICTED: {pred_class}  ({len(subset):,} cases)")
        for _, row in subset.head(3).iterrows():
            print(f"   [Conf: {row['confidence']:.2f}] \"{textwrap.shorten(row['text'], width=120)}\"")
        print()


## 13. Top Predictive Features Analysis
What features drive the model's decisions? We extract the top n-grams from the LinearSVC coefficients.


In [ ]:
feature_names = vectorizer.get_feature_names_out()

def show_top_features(coefs, names, class_labels, top_n=15):
    fig, axes = plt.subplots(1, len(class_labels), figsize=(6*len(class_labels), 6))
    if len(class_labels) == 1:
        axes = [axes]
    for idx, (label, ax) in enumerate(zip(class_labels, axes)):
        coef = coefs[idx]
        top_pos = np.argsort(coef)[-top_n:]
        
        # Color coding: Green for positive weight toward this class
        ax.barh(range(top_n), coef[top_pos], color="#2ecc71")
        ax.set_yticks(range(top_n))
        ax.set_yticklabels([names[i] for i in top_pos], fontsize=10)
        ax.set_title(f"Top drivers for: {label}", fontweight="bold")
        ax.axvline(0, color="black", linewidth=0.5)
        ax.set_xlabel("Coefficient Weight")
        
    plt.suptitle("Highest Weighted Features per Class", fontsize=16, fontweight="bold", y=1.05)
    plt.tight_layout()
    plt.show()

try:
    show_top_features(svc.coef_, feature_names, target_names)
except Exception as e:
    print(f"Feature visualization skipped: {e}")


## 14. Save Model & Vectorizer (Production Artefacts)

Compressing the saved artifacts using `joblib` allows for efficient storage and memory-mapping during inference.


In [ ]:
import joblib

SAVE_DIR = "production_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# Using joblib with compression level 3 (good balance of speed and size)
print("Saving artifacts to disk...")
joblib.dump(vectorizer, os.path.join(SAVE_DIR, "vectorizer.joblib"), compress=3)
joblib.dump(svc, os.path.join(SAVE_DIR, "linear_svc.joblib"), compress=3)
joblib.dump(le, os.path.join(SAVE_DIR, "label_encoder.joblib"), compress=3)

# Calculate sizes
sizes = {fn: os.path.getsize(os.path.join(SAVE_DIR, fn))/1e6
         for fn in os.listdir(SAVE_DIR)}

print("\n✅ Artifacts saved successfully:\n")
for fn, sz in sizes.items():
    print(f"  {fn:25s} : {sz:6.1f} MB")


## 15. Production Inference Pipeline

A standalone function demonstrating how to use the saved artifacts for real-time inference.


In [ ]:
def predict_sentiment(text: str, return_confidence: bool = False):
    """
    End-to-end inference function for a single string.
    """
    # 1. Vectorize
    vec = vectorizer.transform([text])
    
    # 2. Predict
    pred_idx = svc.predict(vec)[0]
    label = le.inverse_transform([pred_idx])[0]
    
    if return_confidence:
        # Get decision function scores
        scores = svc.decision_function(vec)[0]
        # Softmax approximation (for intuition only, not true probabilities)
        exp_scores = np.exp(scores - np.max(scores))
        pseudo_probs = exp_scores / exp_scores.sum()
        conf = pseudo_probs[pred_idx]
        return label, conf
    
    return label

# ── Examples ──
sample_reviews = [
    "Absolutely love this product! The quality is outstanding and delivery was super fast.",
    "Terrible experience. The item broke within 5 minutes of using it. I want a refund.",
    "It arrived on time. The color is exactly as shown in the picture.",
    "The battery life is okay, but the screen has a weird glare. Not sure if I'll keep it.",
    "I was expecting more based on the reviews. It's just average.",
    "WORST CUSTOMER SERVICE EVER. DO NOT BUY FROM THIS SELLER!!!"
]

print("=" * 70)
print(f"{'REVIEW':<55} | {'PREDICTION':<10}")
print("=" * 70)
for review in sample_reviews:
    label, conf = predict_sentiment(review, return_confidence=True)
    short_review = textwrap.shorten(review, width=50, placeholder="...")
    print(f"{short_review:<55} | {label.upper():<10} ({conf:.1%})")
print("=" * 70)


## 16. Classical NLP Tradeoff Analysis

### Why TF-IDF + LinearSVC for this Dataset?

**1. Scalability and Speed:**
- **Training**: Fits 700k+ sparse TF-IDF vectors and trains an SVM in under 2 minutes on a basic CPU.
- **Inference**: Microsecond latency per review. Highly suitable for real-time streaming analytics without requiring GPUs.

**2. Handling Domain Noise (E-commerce):**
- Real reviews are filled with typos ("awsome", "terribleee").
- **Character n-grams** gracefully handle morphological errors without needing expensive spelling correction or subword tokenization models.

**3. The Neutral Class Challenge:**
- The neutral class is typically the hardest to isolate because it often contains mixed sentiments ("Good screen but bad battery").
- **Solution applied**: `class_weight='balanced'` forces the margin objective to penalize neutral-class misclassifications more heavily than positive/negative class errors, vastly improving the Neutral F1 score compared to unweighted models.

**4. vs. Transformer Models (BERT / RoBERTa):**
- A fine-tuned DistilBERT could potentially add 2-5% absolute F1, especially on sarcastic or highly contextual "neutral/mixed" reviews.
- **Tradeoff**: DistilBERT requires GPU inference, increases cloud hosting costs by ~10x, and takes hours to train on 900k rows. LinearSVC offers ~90% of the performance at 1% of the cost.

**Next Optimization Steps for Production:**
1. HPO (Hyperparameter Optimization) on `C` parameter and TF-IDF `min_df` / `max_df` using Optuna.
2. Threshold tuning: Adjusting decision boundaries manually if false positives for a specific class carry higher business cost.

